In [ ]:
import torch
print('PyTorch version:', torch.__version__)
from torch import optim
from torch.utils.data import DataLoader
import numpy as np
from matplotlib import pyplot as plt

#Used in other files
# import os
# from torch import nn
# from PIL import Image
# from torch.utils.data import Dataset, DataLoader
# from torchvision import datasets
# import torchvision.transforms as transforms
# from torchvision.transforms import ToTensor


Load my other files

In [ ]:
from ddpm_config import *
from dataloading import H5ArrayDataset, MCImageDataset
from diffusion_module import diffusion
from Unet_import import UNet
from sparsity import create_sparsity_mask, separate_sparsity_mask #induece sparsity is for sampling

In [ ]:
#initalize objects for training
diffusion_module = diffusion(startVariance=0, maxVariance=diffusion_params['max_variance'], spacing=diffusion_params['schedule_spacing'], diffusionSteps=diffusion_params['num_timesteps'], device=device)
model = UNet(t_dim=UNet_params['t_dim'], device=device)
optimizer = optim.AdamW(model.parameters(), lr = 0.001)

Training Cycle

In [ ]:

#load datasets
image_dataset = MCImageDataset('mc_outputs2')
image_dataloader = DataLoader(image_dataset, batch_size=1, shuffle=True, drop_last=True)

h5_dataset = H5ArrayDataset(h5_path='mc_outputs/processed_arrays2D.h5', dataset_name="images")
h5_dataloader = DataLoader(h5_dataset, batch_size=training_params['batch_size'], shuffle=False)

num_epochs = training_params["num_epochs"]
epoch_losses = []

# plt.axis('off')

#Training loop over all training epochs
for epoch in range(num_epochs):

    #losses is average loss for a given batch
    losses = []

    #now go through every image in the dataset
    for data in h5_dataloader:

        model.train() 

        data = data.to(device) #Shape: B, C, H, W

        #organize variables for the sparsity bits formulation of the diffusion model
        sparsity_mask = create_sparsity_mask(data) #Shape: B, 1, H, W

        data_and_mask = torch.cat([data, sparsity_mask], dim=1) #Shape: B, C+1, H, W
        
        #Create a time vector of shape (B,) will be extended in UNET in time embedding to (B, C+1)
        t = torch.randint(low=0, high=diffusion_params['num_timesteps'], size=(data_and_mask.shape[0],)).to(device)
        
        # data = (data + 1) / 2  # [-1,1] -> [0,1] 
        # plt.imshow(data[2].permute(1,2,0))

        #create noise epsilon of shape data_and_mask: B, C+1, H, W
        noise = torch.randn_like(data_and_mask) 

        #diffused noised data+mask at timestep t
        x_t = diffusion_module.forward(input=data_and_mask, noise=noise, timestep=t)

        # print(x_t.shape)
        # x_t = torch.cat([x_t, -1*torch.ones_like(x_t[:,:1,:,:])], dim = 1)
        # print(x_t.shape)
        # x_t = (x_t + 1) / 2  # [-1,1] -> [0,1] 
        # plt.imshow(x_t[2,:,:,:].permute(1,2,0))

        #get model prediction
        prediction = model(x_t, t)
        predicted_sparsity_mask, predicted_feature_map = separate_sparsity_mask(prediction)

        #claculate losses depending on preferred prediction method specfiied in config
        if training_params['prediction_type'] == 'image':
            feature_map_loss = training_params['fmap_lossfunc'](data, predicted_feature_map)

        elif training_params['prediction_type'] == 'noise':
            feature_map_loss = training_params['fmap_lossfunc'](noise, prediction) #noise is the same shape as the original model prediction, we still want to compare if we preidcted the right noise across the whole map including sparsity
 
        #add sparsity bit losses
        sbit_loss = training_params['sbit_lossfunc'](sparsity_mask, predicted_sparsity_mask)

        #calculate total loss
        batch_loss = sbit_loss+feature_map_loss

        #Reset optimizer for next epoch and perform back propogation
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        #add batch loss
        losses.append(batch_loss.item())

    per_epoch_loss = np.array(losses).mean() #per epoch loss is average of the epoch's batches' losses    
    epoch_losses.append(per_epoch_loss) #list of each epoch's loss

#convert to a tensor for saving in case of possible further training
epoch_losses = torch.tensor(epoch_losses, dtype=torch.float32)

In [ ]:
plt.plot(epoch_losses)

Save Model for Future Training and or Sampling


In [ ]:
model_name = f"ddpm_{str(diffusion_params['schedule_spacing'])}_scheduling_{training_params['prediction_type']}_prediction"
save_dir = 'saved_models'
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'overall_losses': epoch_losses,
    'scheduler_state_dict': diffusion_module.state_dict(),
    }, f'{save_dir}/{model_name}')